# HW1 · Part 4. 소실점 캘리브레이션과 단일 시점 계측 (15점 · 예상 3–4시간)

**배포본:** v5 (2026-10-20)

**목표** — 건물 사진의 직교 3방향 평행선 → 소실점 → 주점 · f (수직심 / Algorithm 8.2) 를 Part 2 의 K 와 비교하고, 기준 높이 하나로 다른 물체의 높이를 재어 줄자와 비교. 정면 사진에서는 무엇이 안 되는지, 주행 프레임의 피치는 어떻게 나오는지

**참고** — MVG 2판 §2.2 (점 · 직선 · 교점), §2.5 (교비), §8.6 (소실점 v = Kd · 소실선 · 직교 관계 · pole–polar), §8.7 Algorithm 8.1 (한 장의 이미지에서 길이비), §8.8 Algorithm 8.2 · Example 8.27 · 8.28 (세 소실점 → K, 초점거리만 모를 때)

**TODO** — `vanishing_point_ls` · `orthocenter` · `f_from_two_vps` · `K_from_three_vps_alg82` · `transfer_height` · `pitch_yaw_from_vp` (6개)

**OpenCV 허용** — `cv2.undistortPoints`(클릭 점의 왜곡 제거), 제공 시각화 코드의 `cv2.undistort`(배경 사진을 같은 좌표계로 표시)

## 4.0 클릭 (노트북 안, 처음 한 번만)
- 건물 사진(`data_demo/building/b1.jpg` 가 예시): 방향별 평행선 4개씩(각 2점; 3개뿐이면 가운데 클릭으로 그 방향을 끝냄) — x · y 는 바닥 타일 · 벽 가로선의 두 방향, **z 는 수직**(문틀 · 벽 모서리). 높이: 기준 물체(문)의 바닥 · 꼭대기, 대상(막대 · 사람)의 바닥 · 꼭대기 — 둘 다 같은 바닥 위
- 정면 사진: 같은 방식으로 방향별 평행선. 주행 프레임: 차선 두 줄(각 2점)
- 왼쪽 클릭 = 점, 오른쪽 클릭 = 취소, 가운데(휠) 클릭 = 현재 방향 끝, 돋보기로 확대 후(해제 후) 클릭. 각 클릭 셀 다음의 저장 셀을 실행 (ipympl 필요: `pip install ipympl`)

> **허용 · 금지** — NumPy · SciPy · Matplotlib 자유. OpenCV 는 각 Part 안내에 "허용" 으로 적힌 함수만(코너 검출 · `findHomography` · 검증용 비교). 
> `TODO` 로 표시된 함수를 OpenCV 호출로 대체하면 해당 항목 0점. 

In [ ]:
# ---- 공통 설정 (HW1 v5 · 모든 노트북에서 같은 설정 파일 사용) ----
import os, sys, json, glob, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'hw1_config.json').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'hw1_config.json').is_file():
    raise FileNotFoundError('HW1 폴더 또는 notebooks 폴더에서 실행하세요: hw1_config.json 을 찾을 수 없습니다.')
sys.path.insert(0, str(ROOT))
from hw1lib import geometry as G, synth as S, exif as E, board as Bd, io_utils as IO, checks as CK, click_tool as CT
# 데모 전환은 HW1/hw1_config.json 의 mode 만 "demo" 로 변경하고 각 노트북의 이 셀을 다시 실행.
cfg = IO.load_config(ROOT)
DATA, RESULTS = cfg['DATA'], cfg['RESULTS']
IO.RESULTS = RESULTS
def load_corners_checked(name):
    """현재 설정과 저장 코너 파일의 검출 배율이 같은지 확인."""
    det = Bd.load_corners(IO.rpath(name))
    if not np.isclose(det['scale'], cfg['SCALE'], rtol=0, atol=1e-12):
        raise ValueError(f"{name}: 저장 배율 {det['scale']}과 현재 설정 scale={cfg['SCALE']}이 다릅니다. Part 0부터 다시 실행하세요.")
    return det
os.makedirs(RESULTS, exist_ok=True)
# 서로 다른 데이터의 결과를 같은 폴더에 섞지 않음. 상대 경로로 기록하여 제출 폴더 이동은 허용.
context = dict(mode=cfg['mode'], data=os.path.relpath(DATA, ROOT), version=5)
context_path = IO.rpath('dataset_context.json')
if os.path.exists(context_path):
    if IO.load_json(context_path) != context:
        raise ValueError('결과 폴더의 데이터 출처가 현재 설정과 다릅니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
elif any(not p.name.startswith('.') for p in Path(RESULTS).iterdir()):
    raise ValueError('출처 기록이 없는 기존 결과가 있습니다. 새 results 경로를 지정하고 Part 0부터 실행하세요.')
IO.save_json(context_path, context)
np.set_printoptions(precision=4, suppress=True)
import matplotlib
_fonts = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
for _f in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR', 'Freesentation']:
    if _f in _fonts:
        matplotlib.rcParams['font.family'] = _f; break
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', message='Glyph')
%matplotlib inline
print('mode =', cfg['mode'], '\nDATA =', DATA, '\nRESULTS =', RESULTS)


In [ ]:
def click_cell(json_path, image, kind, **kw):
    """JSON 이 없을 때만 클릭 도구를 띄움. 반환: clicker 또는 None"""
    if os.path.exists(json_path):
        print('이미 있음:', json_path); return None
    get_ipython().run_line_magic('matplotlib', 'widget')
    return CT.NotebookClicker(image, kind=kind, **kw).show()
def save_cell(clicker, json_path):
    if clicker is not None:
        clicker.save(json_path)
        get_ipython().run_line_magic('matplotlib', 'inline')
J = lambda *p: os.path.join(DATA, *p)
c1 = click_cell(J('building', 'b1_lines.json'), J('building', 'b1.jpg'), 'lines', spec={'x': 4, 'y': 4, 'z': 4}, title='건물 평행선')

In [ ]:
save_cell(c1, J('building', 'b1_lines.json'))
c2 = click_cell(J('building', 'b1_height_points.json'), J('building', 'b1.jpg'), 'points', names=['b_ref', 't_ref', 'b_x', 't_x'], title='높이')

In [ ]:
save_cell(c2, J('building', 'b1_height_points.json'))
c3 = click_cell(J('building', 'b2_frontal_lines.json'), J('building', 'b2_frontal.jpg'), 'lines', spec={'x': 4, 'y': 4, 'z': 4}, title='정면 사진 평행선')

In [ ]:
save_cell(c3, J('building', 'b2_frontal_lines.json'))
c4 = click_cell(J('driving', 'frame_lines.json'), J('driving', 'frame.png'), 'lines', spec={'left': 1, 'right': 1}, title='차선')

In [ ]:
save_cell(c4, J('driving', 'frame_lines.json'))

In [ ]:
# Part 2의 K를 대상 이미지의 픽셀 격자로 변환한 뒤 클릭점을 역왜곡.
import cv2
K_cal, kinfo = IO.load_K('K.json'); dist_z, _ = IO.load_dist('dist.json')
def image_size_of(path):
    h, w = E.load_image(path, gray=True).shape[:2]
    return (w, h)
def K_for_image(path):
    return IO.rescale_K_for_image(K_cal, kinfo['image_size'], image_size_of(path))
def undistort_uv(uv, K_image):
    uv = np.asarray(uv, float).reshape(-1, 1, 2)
    return cv2.undistortPoints(uv, K_image, dist_z, P=K_image).reshape(-1, 2)
print('K · dist 로드:', kinfo['source'], '캘리브레이션 크기:', kinfo['image_size'])


## 4.1 소실점 — 직선의 최소제곱 교점 (3점)

In [ ]:
# ==============================================================================
# TODO  vanishing_point_ls
# 참조: MVG §2.2.1 (두 직선의 교점 x = l × l′, 두 점을 잇는 직선 l = x × x′), §8.6.1 (소실점 v = Kd)
# 직선 2개면 외적으로 끝. k ≥ 3 개면 Σ_i (l_iᵀ v)² 를 최소화하는 v (‖v‖ = 1) → L = [l_1ᵀ; …; l_kᵀ] 의 최소 특이값 우특이벡터
# 직선을 (a, b) 노름으로 나눠 두면 l_iᵀ v 가 "점 v 까지의 거리 × w" 가 되어 가중이 공평해짐
# 반환: 동차 3-벡터 v (무한원점에 가까우면 v[2] ≈ 0 — 정규화하지 말고 그대로 돌려줄 것)
# ==============================================================================
def vanishing_point_ls(lines):
    """직선들 (k, 3) (동차, l = p × q) 의 최소제곱 교점: Σ(lᵢᵀv)² 최소 → Σ lᵢlᵢᵀ 의 최소 고유벡터
    (MVG §2.2.1 두 직선의 교점, §8.6.1 소실점). 직선은 (a, b) 노름으로 정규화해 거리 가중을 맞춤."""
    # ---- 여기에 작성 ----
    L = np.atleast_2d(np.asarray(lines, float))
    L = L / np.linalg.norm(L[:, :2], axis=1, keepdims=True)   # (a, b) 노름 = 1 → lᵀv 가 거리 × w
    _, _, Vt = np.linalg.svd(L, full_matrices=True)           # Σ(lᵢᵀv)² 최소, ‖v‖ = 1
    v = Vt[-1]
    if v[2] < 0:                                              # 부호만 정리 (무한원점이면 v[2] ≈ 0 그대로)
        v = -v
    return v


In [ ]:
def load_lines(path):
    rec = IO.load_json(path)
    image_path = os.path.join(os.path.dirname(path), rec['image'])
    size = image_size_of(image_path); K_image = K_for_image(image_path)
    out = {}
    for key, segs in rec['lines'].items():
        segs = [undistort_uv(np.array(s, float), K_image) for s in segs]
        out[key] = [(s[0], s[1]) for s in segs]
    return rec, out, K_image, size
rec, lines, K_z, building_image_size = load_lines(os.path.join(DATA, 'building', 'b1_lines.json'))
if not all(k in lines and len(lines[k]) >= 2 for k in 'xyz'):
    raise ValueError('x, y, z 방향마다 직선 2개 이상이 필요합니다.')
vps = {}
for key in ['x', 'y', 'z']:
    L = np.array([G.line_through(p, q) for p, q in lines[key]])
    v = vanishing_point_ls(L); vps[key] = v
    vh = G.hnorm(v)
    dists = [abs(l @ vh) / np.hypot(l[0], l[1]) for l in L]   # 소실점에서 각 직선까지의 거리 (px) — 일관성 지표
    print(f'{key}: v = ({vh[0]:.1f}, {vh[1]:.1f}), w = {v[2]/np.linalg.norm(v[:2]):.2e},  직선까지 거리 최대 {max(dists):.2f} px')
img_raw = E.load_image(os.path.join(DATA, 'building', rec['image']))[:, :, ::-1]
img = cv2.undistort(img_raw, K_z, dist_z)  # 선분과 배경 모두 같은 보정 픽셀 좌표
fig, ax = plt.subplots(figsize=(12, 8)); ax.imshow(img)
for key, col in zip('xyz', 'rgb'):
    for p, q in lines[key]:
        ax.plot([p[0], q[0]], [p[1], q[1]], '-', color=col, lw=1.5)
    vh = G.hnorm(vps[key]); ax.plot(vh[0], vh[1], 'o', color=col, ms=8, mfc='none', mew=2)
ax.set_xlim(min(0, min(G.hnorm(v)[0] for v in vps.values()) - 50), max(img.shape[1], max(G.hnorm(v)[0] for v in vps.values()) + 50))
ax.set_ylim(max(img.shape[0], max(G.hnorm(v)[1] for v in vps.values()) + 50), min(0, min(G.hnorm(v)[1] for v in vps.values()) - 50))
ax.set_title('소실점 (x 빨강 · y 초록 · z 파랑)'); fig.savefig(IO.rpath('part4_vps.png'), dpi=100)
CK.check_vanishing_point(vanishing_point_ls);

## 4.2 주점과 f — 수직심 · 두 소실점의 내적 · Algorithm 8.2 (4점)
세 직교 소실점의 삼각형에서 수직심 = 주점(MVG Example 8.27 · Fig. 8.23), (v₁ − p)·(v₂ − p) = −f² (Fig. 8.24). Algorithm 8.2 형태(ω 4 미지수)로도 풀어 두 결과와 Part 2 의 K 를 비교합니다.

In [ ]:
# ==============================================================================
# TODO  orthocenter
# 참조: MVG §8.8 Example 8.27 (직교하는 세 소실점 — 주점은 소실점 삼각형의 수직심), Fig. 8.23
# 직교하는 세 방향의 소실점 v1, v2, v3 가 만드는 삼각형의 수직심(세 높이선의 교점)이 주점
# v1 을 지나고 변 v2v3 에 수직인 직선: 방향 d = v3 − v2, 직선 = (d_x, d_y, −d·v1).  두 높이선의 외적 → 교점 (정규화)
# 반환 (2,) 픽셀.  소실점 하나가 무한대 근처면 수직심이 불안정 — Part 4.4 에서 체험
# ==============================================================================
def orthocenter(v1, v2, v3):
    """세 소실점 삼각형의 수직심 = 주점 (MVG Example 8.27 · Fig. 8.23)."""
    # ---- 여기에 작성 ----
    def _p(v):
        v = np.asarray(v, float).ravel()
        return G.hnorm(v)[:2] if v.size == 3 else v[:2]
    a, b, c = _p(v1), _p(v2), _p(v3)
    d1 = c - b
    l1 = np.array([d1[0], d1[1], -d1 @ a])               # v1 을 지나고 변 v2v3 에 수직
    d2 = c - a
    l2 = np.array([d2[0], d2[1], -d2 @ b])               # v2 를 지나고 변 v1v3 에 수직
    x = np.cross(l1, l2)
    return x[:2] / x[2]


# ==============================================================================
# TODO  f_from_two_vps
# 참조: MVG §8.8 Example 8.27 · Fig. 8.24 (초점거리), Example 8.28 (초점거리만 모를 때)
# 직교 방향의 두 소실점 v1, v2 와 주점 p:  (v1 − p)·(v2 − p) = −f²   (주점을 원점으로 두면 ω = diag(1/f², 1/f², 1) 이고 v1ᵀ ω v2 = 0 에서 x1x2 + y1y2 + f² = 0)
# 내적이 양수면 f² < 0 → 직교 가정이 깨짐(또는 소실점 오류) → NaN 반환
# ==============================================================================
def f_from_two_vps(v1, v2, pp):
    """f² = −(v1 − p)·(v2 − p)  (MVG Example 8.27 · Fig. 8.24)."""
    # ---- 여기에 작성 ----
    def _p(v):
        v = np.asarray(v, float).ravel()
        return G.hnorm(v)[:2] if v.size == 3 else v[:2]
    a, b, p = _p(v1), _p(v2), np.asarray(pp, float).ravel()[:2]
    f2 = -float((a - p) @ (b - p))                        # (v1 − p)·(v2 − p) = −f²
    return float(np.sqrt(f2)) if f2 > 0 else float('nan')


# ==============================================================================
# TODO  K_from_three_vps_alg82
# 참조: MVG §8.8 Algorithm 8.2, Table 8.1 (ω 에 대한 제약)
# skew 0 · 정사각 픽셀 가정 → ω = [[w1, 0, w2], [0, w1, w3], [w2, w3, w4]]  (미지수 4)
# 제약 v_iᵀ ω v_j = 0 (i < j, 3개) 를 w = (w1, w2, w3, w4) 에 대한 1차식으로 쓰면 A w = 0 (3 × 4) → SVD 최소 특이벡터
#   v_iᵀ ω v_j = w1 (a0 b0 + a1 b1) + w2 (a0 b2 + a2 b0) + w3 (a1 b2 + a2 b1) + w4 a2 b2   (a = v_i, b = v_j, 동차 좌표)
# ω → K: ω = (K Kᵀ)⁻¹ → ω⁻¹ = K Kᵀ 를 "상삼각 K" 로 Cholesky 분해.  np.linalg.cholesky 는 하삼각 L Lᵀ 를 주므로
#   순열 P = 역순 단위행렬로 뒤집어 L = chol(P ω⁻¹ P), K = P L P (그 뒤 K[2,2] = 1 로 정규화).  ω[0,0] < 0 이면 ω ← −ω
# 반환 (K, 특이값).  세 소실점만 쓰면 3 식 · 4 미지수(동차) → 정확히 결정 — 잡음에 민감
# ==============================================================================
def K_from_three_vps_alg82(v1, v2, v3):
    """Algorithm 8.2 형태 (MVG §8.8): ω = [w1 0 w2; 0 w1 w3; w2 w3 w4] (skew 0 · 정사각 픽셀, 미지수 4)
    제약 vᵢᵀ ω vⱼ = 0 (i<j) 3개 → Aw = 0 → w → ω → K = chol(ω⁻¹) (상삼각, 대각 양)."""
    # ---- 여기에 작성 ----
    V = []
    for v in (v1, v2, v3):
        v = np.asarray(v, float).ravel()
        v = np.append(v, 1.0) if v.size == 2 else v
        V.append(v / np.linalg.norm(v))
    A = np.array([[a[0] * b[0] + a[1] * b[1], a[0] * b[2] + a[2] * b[0], a[1] * b[2] + a[2] * b[1], a[2] * b[2]]
                  for a, b in [(V[0], V[1]), (V[0], V[2]), (V[1], V[2])]])
    _, s, Vt = np.linalg.svd(A, full_matrices=True)
    w1, w2, w3, w4 = Vt[-1]
    omega = np.array([[w1, 0.0, w2], [0.0, w1, w3], [w2, w3, w4]])
    if omega[0, 0] < 0:
        omega = -omega
    Pm = np.eye(3)[::-1]                                  # 역순 순열
    L = np.linalg.cholesky(Pm @ np.linalg.inv(omega) @ Pm)   # ω⁻¹ 가 양정치가 아니면 LinAlgError
    K = Pm @ L @ Pm                                       # 상삼각, 대각 양, K Kᵀ = ω⁻¹
    return K / K[2, 2], s


In [ ]:
v1, v2, v3 = [G.hnorm(vps[k])[:2] for k in 'xyz']
pp = orthocenter(v1, v2, v3)
fs = [f_from_two_vps(a, b, pp) for a, b in [(v1, v2), (v1, v3), (v2, v3)]]
K82, s82 = K_from_three_vps_alg82(vps['x'], vps['y'], vps['z'])
print('수직심(주점) = (%.1f, %.1f),  f (세 쌍) = %s' % (pp[0], pp[1], np.round(fs, 1)))
print('Algorithm 8.2: f = %.1f, c = (%.1f, %.1f)  (특이값 %s)' % (K82[0, 0], K82[0, 2], K82[1, 2], np.round(s82, 3)))
if K_z is not None:
    print('Zhang(C): f = %.1f / %.1f, c = (%.1f, %.1f)  → 소실점 f 의 차이 %+.1f %%, 주점 차이 (%.1f, %.1f) px' % (K_z[0, 0], K_z[1, 1], K_z[0, 2], K_z[1, 2], (np.nanmean(fs) / K_z[0, 0] - 1) * 100, pp[0] - K_z[0, 2], pp[1] - K_z[1, 2]))
IO.save_json(IO.rpath('vp.json'), dict(vps={k: vps[k] for k in 'xyz'}, pp=pp, f=fs, f_mean=float(np.nanmean(fs)), K_alg82=K82, image=rec['image'], image_size=list(building_image_size)))
CK.check_three_vps(orthocenter, f_from_two_vps, K_from_three_vps_alg82);

## 4.3 단일 시점 높이 — Algorithm 8.1 (4점)
지면 소실선은 두 가지로: (i) 수평 두 소실점을 잇는 직선 (MVG §8.6.2), (ii) 수직 소실점의 polar l = ω v (§8.6.3 식 (8.17), ω = (KKᵀ)⁻¹ 에 Part 2 의 K). 두 소실선의 차이도 보고하세요. 교비의 거리는 **부호 포함**.

In [ ]:
# ==============================================================================
# TODO  transfer_height
# 참조: MVG §8.7 Algorithm 8.1 (한 장의 이미지에서 장면의 길이비), §2.5 교비
# 입력: 기준 물체의 바닥 b_ref · 꼭대기 t_ref (높이 H_ref), 대상의 b_x · t_x, 수직 소실점 v, 지면 소실선 horizon (동차 3-벡터)
# ① 바닥점 둘을 잇는 직선 (b_ref × b_x) 과 소실선의 교점 u — 지면 위 "두 바닥을 잇는 방향" 의 소실점
# ② u 와 t_ref 를 잇는 직선 (높이 H_ref 의 평행선의 상) 이 대상의 수직선 (b_x × v) 과 만나는 점 t̃ — 대상 위치로 옮겨진 기준 높이
# ③ 수직선 위 네 점 b_x, t̃, t_x, v 의 교비:  H_x / H_ref = d(b_x, t_x) · d(t̃, v) / ( d(b_x, t̃) · d(t_x, v) )
#    d 는 수직선 위의 "부호 포함" 거리(한 방향을 + 로 잡고 재기) — v 가 점들 반대편에 있어도 부호가 맞아야 함
#    v 가 무한원점(v[2] ≈ 0)이면 d(·, v) 의 비는 1 → 단순 길이비 (아핀).  반환 H_x (float, H_ref 와 같은 단위)
# ==============================================================================
def transfer_height(b_ref, t_ref, b_x, t_x, v, horizon, H_ref):
    """Algorithm 8.1 (MVG §8.7): 기준 (b_ref, t_ref, 높이 H_ref) 로 대상 (b_x, t_x) 의 높이.
    1) 두 바닥점을 잇는 직선과 소실선 l 의 교점 u  2) u 와 t_ref 를 잇는 직선이 대상의 수직선 (b_x, v) 와 만나는 점 t̃
    3) 수직선 위 네 점 b_x, t̃, t_x, v 의 교비:  H_x/H_ref = d(b_x, t_x) d(t̃, v) / (d(b_x, t̃) d(t_x, v))  — d 는 직선 위 부호 포함 거리
    v 가 무한원점(w ≈ 0)이면 d(·, v) 비는 1."""
    # ---- 여기에 작성 ----
    def _h(p):
        p = np.asarray(p, float).ravel()
        return np.append(p[:2], 1.0) if p.size == 2 else p
    bR, tR, bX, tX = (_h(p) for p in (b_ref, t_ref, b_x, t_x))
    v = np.asarray(v, float).ravel()
    l = np.asarray(horizon, float).ravel()
    u = np.cross(np.cross(bR, bX), l)                     # ① 두 바닥을 잇는 직선 ∩ 소실선
    l_vert = np.cross(bX, v)                              # 대상의 수직선
    t_tl = np.cross(np.cross(u, tR), l_vert)              # ② 기준 높이를 대상 위치로 옮긴 점 t̃
    b = bX[:2] / bX[2]
    v_inf = abs(v[2]) < 1e-9 * np.linalg.norm(v[:2])
    if v_inf:
        e = v[:2] / np.linalg.norm(v[:2])
    else:
        vp = v[:2] / v[2]
        e = (vp - b) / np.linalg.norm(vp - b)             # b → v 방향을 + 로
    coord = lambda p: (p[:2] / p[2] - b) @ e              # 수직선 위 부호 포함 1D 좌표 (b = 0)
    s_tt, s_tx = coord(t_tl), coord(tX)
    if v_inf:                                             # 아핀: 단순 길이비
        return float(H_ref * s_tx / s_tt)
    s_v = np.linalg.norm(vp - b)
    ratio = (s_tx * (s_v - s_tt)) / (s_tt * (s_v - s_tx))   # ③ d(b,t_x) d(t̃,v) / (d(b,t̃) d(t_x,v))
    return float(H_ref * ratio)


In [ ]:
meas = IO.load_json(IO.rpath('measurements.json'))
hp = IO.load_json(os.path.join(DATA, 'building', 'b1_height_points.json'))
if hp['image'] != rec['image'] or meas['height_image'] != rec['image']:
    raise ValueError('높이 실측값·높이 클릭점·소실점이 같은 사진이어야 합니다. height_image와 JSON의 image를 확인하세요.')
pts = undistort_uv(np.array(hp['points'], float), K_z); b_ref, t_ref, b_x, t_x = pts
v_vert = vps['z']
horizon_i = np.cross(G.hnorm(vps['x']), G.hnorm(vps['y']))
H_ref = meas['ref_height_mm']; H_meas = meas.get('target_height_mm')
if H_ref is None or not np.isfinite(H_ref) or H_ref <= 0:
    raise ValueError('ref_height_mm에 기준 물체의 양수 높이(mm)를 입력하세요.')
H_i = transfer_height(b_ref, t_ref, b_x, t_x, v_vert, horizon_i, H_ref)
out = dict(H_ref=H_ref, H_est_horizon_from_vps=H_i, H_measured=H_meas)
print('(i) 두 소실점의 소실선: 대상 높이 %.1f mm' % H_i + (' (실측 %.1f, 오차 %+.1f %%)' % (H_meas, (H_i / H_meas - 1) * 100) if H_meas else ''))
if K_z is not None:
    omega = np.linalg.inv(K_z @ K_z.T)
    horizon_ii = omega @ G.hnorm(v_vert)
    H_ii = transfer_height(b_ref, t_ref, b_x, t_x, v_vert, horizon_ii, H_ref)
    ang = np.degrees(np.arccos(np.clip(abs(horizon_i[:2] @ horizon_ii[:2]) / np.linalg.norm(horizon_i[:2]) / np.linalg.norm(horizon_ii[:2]), 0, 1)))
    print('(ii) ω v 소실선: 대상 높이 %.1f mm, 두 소실선의 각도 차 %.2f°' % (H_ii, ang))
    out.update(H_est_horizon_omega=H_ii, horizon_angle_diff_deg=ang)
IO.save_json(IO.rpath('height.json'), out)
CK.check_transfer_height(transfer_height);

## 4.4 정면 사진 — 무엇이 안 되는가 (2점)
한 방향이 광축과 평행하면 그 소실점은 주점 근처, 나머지 둘은 무한원점 근처(w ≈ 0). 수직심 · f 가 어떻게 되는지 숫자로 보이고, **MVG Example 8.28 (초점거리만 모를 때 · 정확도)** 과 연결해 설명하세요.

In [ ]:
rec2, lines2, K_front, frontal_image_size = load_lines(os.path.join(DATA, 'building', 'b2_frontal_lines.json'))
assert all(len(lines2[k]) >= 2 for k in 'xyz'), '방향마다 직선 2개 이상을 클릭하세요: ' + str({k: len(v) for k, v in lines2.items()})
vps2 = {k: vanishing_point_ls(np.array([G.line_through(p, q) for p, q in lines2[k]])) for k in 'xyz'}
for k in 'xyz':
    v = vps2[k]; print(f'{k}: w/|v_xy| = {v[2]/np.linalg.norm(v[:2]):.2e}  →  v ≈ ({G.hnorm(v)[0]:.0f}, {G.hnorm(v)[1]:.0f})')
with np.errstate(all='ignore'):
    pp2 = orthocenter(*[G.hnorm(vps2[k])[:2] for k in 'xyz'])
    f2 = [f_from_two_vps(G.hnorm(vps2[a])[:2], G.hnorm(vps2[b])[:2], pp2) for a, b in [('x', 'y'), ('x', 'z'), ('y', 'z')]]
print('수직심 =', pp2, ' f (세 쌍) =', np.round(f2, 1))
try:
    K82_2, s82_2 = K_from_three_vps_alg82(vps2['x'], vps2['y'], vps2['z'])
    print('Algorithm 8.2 특이값 =', np.round(s82_2, 4), ' K =\n', K82_2)
except (ValueError, np.linalg.LinAlgError) as e:
    print('Algorithm 8.2 실패:', e)

## 4.5 주행 프레임 — 차선 소실점 → 피치 · 요 · 지평선 (2점, 9월 HW1 Part C)
제공 프레임의 K · 카메라 높이는 `data/driving/frame_K.json`. 차선 두 줄을 클릭해 전방 소실점을 구하고 피치 · 요 · 지평선을 그립니다.
피치와 높이로 **지면 위 점까지의 거리** 도 구할 수 있습니다: 이미지 아래쪽 행 v 의 지면 거리 d = h / tan(θ + atan((v − c_y)/f_y)), θ = 아래로 숙인 각(양수). `pitch_yaw_from_vp` 의 pitch 는 아래를 보면 **음수**이므로 θ = −pitch.

In [ ]:
# ==============================================================================
# TODO  pitch_yaw_from_vp
# 참조: MVG §8.6.1 Result 8.20 (소실점 v = Kd, 월드 방향이면 v = KRd)
# 전방 소실점 v = K R d (d = 차량 전방) → 카메라 좌표의 전방 방향 d_c = K⁻¹ v / ‖K⁻¹ v‖
# pitch = atan2(d_y, d_z) (도): 카메라가 아래를 보면 지평선 · 소실점이 주점 위 → d_y < 0 → 음수 (카메라가 1.86° 아래로 기울었으면 pitch = −1.86°).  yaw = atan2(d_x, d_z) (오른쪽 양수)
# 반환 (pitch_deg, yaw_deg).  ※ 롤은 소실점 하나로는 안 나옴 (지평선의 기울기가 필요)
# ==============================================================================
def pitch_yaw_from_vp(K, vp):
    """전방 소실점 v = K R d (d = 전방 방향) 에서 피치 · 요 (MVG Result 8.20 v = Kd).
    d_c = K⁻¹ v 를 단위 벡터로 → 카메라 좌표에서 전방 방향. pitch = atan2(d_y, d_z) (아래를 보면 소실점이 주점 위 → d_y < 0 → 음수;
    카메라가 1.86° 아래로 기울었으면 이 규약으로 pitch = −1.86°),  yaw = atan2(d_x, d_z) (오른쪽 양수)."""
    # ---- 여기에 작성 ----
    vp = np.asarray(vp, float).ravel()
    if vp.size == 2:
        vp = np.append(vp, 1.0)
    d = np.linalg.solve(np.asarray(K, float), vp)         # d_c ∝ K⁻¹ v
    d = d / np.linalg.norm(d)
    if d[2] < 0:                                          # 전방 (Z_c > 0)
        d = -d
    pitch = np.degrees(np.arctan2(d[1], d[2]))
    yaw = np.degrees(np.arctan2(d[0], d[2]))
    return float(pitch), float(yaw)


In [ ]:
info = IO.load_json(os.path.join(DATA, 'driving', 'frame_K.json')); Kd = np.array(info['K']); h_cam = info['camera_height_m']
recd = IO.load_json(os.path.join(DATA, 'driving', 'frame_lines.json'))
L = np.array([G.line_through(np.array(s[0]), np.array(s[1])) for segs in recd['lines'].values() for s in segs])
vd = G.hnorm(vanishing_point_ls(L))
pitch, yaw = pitch_yaw_from_vp(Kd, vd)
print('전방 소실점 (%.1f, %.1f) → pitch %.2f°, yaw %.2f°' % (vd[0], vd[1], pitch, yaw))
img = E.load_image(os.path.join(DATA, 'driving', recd['image']))[:, :, ::-1]
fig, ax = plt.subplots(figsize=(11, 6.5)); ax.imshow(img)
for segs in recd['lines'].values():
    for s in segs:
        ax.plot([s[0][0], s[1][0]], [s[0][1], s[1][1]], 'y-', lw=1.5)
ax.axhline(vd[1], color='c', lw=1, ls='--', label='지평선 (롤 0 가정)'); ax.plot(vd[0], vd[1], 'ro', label='소실점')
for v_row in (650, 500, 420):
    theta = np.radians(-pitch) + np.arctan((v_row - Kd[1, 2]) / Kd[1, 1])
    d_ground = h_cam / np.tan(theta) if theta > 0 else np.inf
    ax.axhline(v_row, color='w', lw=0.6); ax.text(10, v_row - 5, f'row {v_row}: {d_ground:.1f} m', color='w')
ax.legend(loc='upper right'); ax.set_axis_off(); fig.savefig(IO.rpath('part4_driving.png'), dpi=100)
IO.save_json(IO.rpath('driving.json'), dict(vp=vd[:2], pitch_deg=pitch, yaw_deg=yaw, horizon_row=vd[1]))
CK.check_pitch(pitch_yaw_from_vp);

## 보고서에 넣을 것 (Part 4, 슬라이드 4장)
1. 선분 · 소실점 그림 + 소실점 표(좌표 · w · 직선까지 거리)
2. 주점 · f: 수직심 vs Algorithm 8.2 vs Zhang — 표와 차이 %, 왜 다른지 한 문단(클릭 오차의 전파 · 직교 가정 · 왜곡 잔차)
3. 높이: 그림(점 · 소실선 · t̃) + 결과(두 소실선 방식) vs 실측
4. 정면 사진에서 생긴 일(숫자) + 주행 프레임 피치 · 지평선 · 거리 눈금

In [ ]:
CK.run_all(globals());